## Ejercicio 5. (*)

Se define el método **MXY** mediante la siguiente expresión:

$$
y_k = x_k - \frac{2}{3}\frac{f(x_k)}{f'(x_k)}
$$

$$
x_{k+1} = x_k - \frac{f(x_k)}{f'(y_k)} \cdot \frac{5f'(y_k) - f'(x_k)}{4f'(y_k)}

$$

### a) (1.0 pts)

Programar el método del **MXY** en Python.

In [9]:
import math

def MXY(f, df, x0, tol, maxiter):
    iters = 0
    error = tol + 1

    while error > tol and iters <= maxiter:
        df_x0 = df(x0)
        if df_x0 == 0 or math.isnan(df_x0):
            print("Error: Derivada nula o no válida en x0")
            return [x0, iters, error]

        y = x0 - (2 / 3) * (f(x0) / df_x0)
        
        df_y = df(y)
        if df_y == 0 or math.isnan(df_y):
            print("Error: Derivada nula o no válida en y")
            return [x0, iters, error]

        x = x0 - (f(x0) / df_y) * ((5 * df_y - df_x0) / (4 * df_y))
        
        if math.isnan(x) or math.isinf(x):
            print("El método divergió (se obtuvo NaN o Inf)")
            return [x0, iters, error]

        error = math.fabs(x - x0)
        iters += 1
        x0 = x

    return [x, iters, error]



### b) (1.0 pts)

Utilizar el método anterior para resolver la ecuación:

$$
e^x\cos(x) - x^3\sin(x) - 1 = 0
$$

usando:

- Estimación inicial: $x_0 = 3$
- Tolerancia: $10^{-7}$

In [10]:
f = lambda x: math.exp(x) * math.cos(x) - x**3 * math.sin(x) - 1
df = lambda x: (math.exp(x) * (math.cos(x) - math.sin(x)) - (x**2) * (3 * math.sin(x) + x * math.cos(x)))

x0 = 3
tol = 1e-7
maxiter = 30

x, iters, error = MXY(f, df, x0, tol, maxiter)

print(f"La solución a la ecuación de x = {x}")
print(f"Las iteraciones realizadas fueron {iters}")
print(f"El error que se tuvo fue de {error}")

La solución a la ecuación de x = 3.0
Las iteraciones realizadas fueron 1
El error que se tuvo fue de 0.0


Comparar los resultados con los métodos de **Newton, Traub y Ostrowski**.

In [11]:
def Newton(f, df, x0, tol, maxiter):
    iters = 0
    error = tol+1

    while error > tol and iters <= maxiter:
        x = x0-f(x0)/df(x0)
        error = math.fabs(x-x0)
        iters = iters+1
        x0 = x
    return [x, iters, error]

In [12]:
def Traub(f,df,x0,tol,maxiter):
    iters = 0
    error = tol + 1

    while iters <= maxiter and error > tol:
        y = x0-f(x0)/df(x0)
        x = y-f(y)/df(x0)
        error = math.fabs(x-x0)
        iters += 1
        x0 = x
    return [x,iters, error]

In [13]:
import numpy as np

def Ostrowski(f,df,x0,tol,maxiter):
    iters = 0
    error = tol+1
    error_x = list()

    while error>tol and iters<=maxiter:
        y = x0-f(x0)/df(x0)
        x = y-f(x0)/(f(x0)-2*f(y))*f(y)/df(x0)
        error = math.fabs(x-x0)
        error_x.append(error)
        iters = iters+1
        x0 = x

    error_x = np.array(error_x)
    eps = 1e-16
    error_x_safe = np.maximum(error_x, eps)
    ACOC = np.log(error_x_safe[2:]/error_x_safe[1:-1])/np.log(error_x_safe[1:-1]/error_x_safe[0:-2])
    return [x,iters,error,ACOC]

In [14]:
x, iters, error = Newton(f, df, x0, tol, maxiter)

print("=====Newton=====")
print(f"La solución obtenida fue {x}")
print(f"La cantidad de iteracione realizadas fueron {iters}")
print(f"El error que se obtuvo fue de {error}")

=====Newton=====
La solución obtenida fue 124.09290981679683
La cantidad de iteracione realizadas fueron 5
El error que se obtuvo fue de 5.3943338684803166e-08


In [15]:
x, iters, error = Traub(f, df, x0, tol, maxiter)

print("=====Traub=====")
print(f"La solución obtenida fue {x}")
print(f"La cantidad de iteracione realizadas fueron {iters}")
print(f"El error que se obtuvo fue de {error}")

=====Traub=====
La solución obtenida fue -2.6713743274964937e+55
La cantidad de iteracione realizadas fueron 2
El error que se obtuvo fue de 0.0


In [16]:
x, iters, error, ACOC = Ostrowski(f, df, x0, tol, maxiter)

print("=====Ostrowski=====")
print(f"La solución obtenida fue {x}")
print(f"La cantidad de iteracione realizadas fueron {iters}")
print(f"El error que se obtuvo fue de {error}")

=====Ostrowski=====
La solución obtenida fue 64.40264939859077
La cantidad de iteracione realizadas fueron 4
El error que se obtuvo fue de 0.0


### Comparación de resultados

| Método | Resultado | Iteraciones | Error |
|---|---|---|---|
| MXY | $3.0$ | 1 | $0$ |
| Newton | $124.0929098168$ | 5 | $5.39\times10^{-8}$ |
| Traub | $-2.67\times10^{55}$ | 2 | $0$ |
| Ostrowski | $64.4026493986$ | 4 | $0$ |

- **MXY:** se queda en $x = 3$ con error $0$, pero $f(3) \approx -24.69$, por lo que no es una raíz. El punto $y_0$ salta a $85.09$, donde $f'(y_0) \approx -6.3\times10^{36}$; la corrección resulta tan pequeña que $x_1 = x_0$ y el criterio $|x_{k+1} - x_k|$ da una falsa convergencia.
- **Traub:** diverge a $-2.67\times10^{55}$; el error $0$ solo indica que el valor ya no cambia en aritmética de punto flotante.
- **Newton y Ostrowski:** convergen a raíces reales de la ecuación, pero muy alejadas de la estimación inicial.

**Conclusión:** $x_0 = 3$ es una mala estimación inicial para los cuatro métodos, porque está junto a un punto crítico de $f$: $f'(x) = 0$ en $x \approx 2.9959$ y $f'(3) \approx 0.20$. Al dividir entre una derivada casi nula, el primer paso es enorme ($f(3)/f'(3) \approx -123$) y ningún método alcanza la raíz más cercana, $x \approx 3.8433762366$.